# COS30049 A2 - Round 2 pretraining tokens for the 152M model (Kaggle CPU session)

Continues round 1's FineWeb-Edu corpus (`docs/KAGGLE_LM_PLAN.md`, section 4, C4) on a CPU session, which uses no
GPU quota: `python -m lm.prepare_text --continue-from` starts at the document after round 1's last one (file 2, row
483,432 of the 10B-token sample), with round 1's tokenizer and validation tokens, so no document is trained on twice
and both rounds' validation losses are measured on the same tokens.

**Settings (right-hand panel):** Accelerator **None**, **Internet on**. **Input:** this account's dataset
`cos30049a2-text150m-base` (round 1's tokenizer, `val.bin`, `meta.json` and `state.json`, with SHA-256). Then **Save
Version -> Save & Run All (Commit)**. Output: `text_cont/` (about 2 bytes per token) and `data_manifest.json`, read by
`cos30049a2-text150m-train`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "e8174753bf7c39ad0673f312e9365756c97b0022"   # the code this build uses; recorded in data_manifest.json
TOKENS = 3_000_000_000  # new training tokens: more than the GPU quota can train this round, so the gate decides
PINS = ["numpy==2.2.6", "tokenizers==0.21.1", "huggingface_hub==0.30.2", "pyarrow==20.0.0"]  # environment.yml

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
base = check_base()
(SRC / "data" / "text").mkdir(parents=True, exist_ok=True)
for name in ("tokenizer.json", "val.bin", "meta.json", "state.json"):
    shutil.copy(base / name, SRC / "data" / "text" / name)
OUT = WORK / "text_cont"
sh(sys.executable, "-m", "lm.prepare_text", "--continue-from", "data/text", "--out", str(OUT), "--tokens", str(TOKENS),
   "--raw", "/tmp/fineweb-edu", "--delete-raw", log="prepare-text-cont.log")

In [ ]:
meta = json.loads((OUT / "meta.json").read_text(encoding="utf-8"))
manifest = {"commit": COMMIT, "built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS},
            "train_tokens": meta["train_tokens"], "continued_from": meta["continued_from"],
            "files": {f"text_cont/{f.name}": sha256(f) for f in sorted(OUT.iterdir()) if f.is_file()}}
(WORK / "data_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: v for k, v in manifest.items() if k != "files"}, indent=2))